# 01_03 Tags and entities: what role does each word play?

Knowing the words is not knowing who did what to whom. This notebook tags each word with its part of
speech, groups words into phrases, and finds the names, places, dates and amounts in Kittiwake's notices.
Along the way you will reproduce a result from the book's own notebook, where a college was labelled a
PERSON, and find out exactly why.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-01-why-cant-a-computer-read", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'spacy': 'spacy',
           'nltk': 'nltk',
           'transformers': 'transformers',
           'en_core_web_sm': 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
from collections import Counter
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.util import ngrams, skipgrams
from nlpcheck import ask, guess, reveal, check_01_03

nlp = spacy.load("en_core_web_sm")
notices = open("data/kittiwake_notices.txt").read().strip().split("\n\n")
sw = set(stopwords.words("english"))

## 1. Recall

**r5.** Unless you tell it otherwise, which part of speech does NLTK's `WordNetLemmatizer` assume?
(one word)

**r6.** Why did the stop list turn "not working" into "working"?
(a) the tokenizer dropped it, (b) "not" is on NLTK's stop list, (c) lemmatization removed it

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. Parts of speech

A **part-of-speech tagger** labels each token as a noun, verb, adjective and so on. NLTK's tagger uses
the **Penn Treebank** tag set (`NN` noun, `VB` verb, `JJ` adjective, `NNP` proper noun, ...). It decides
from the word and the words around it, because the same word can be different parts of speech.

Predict the tag NLTK gives `book` in "Please book the engineer for Friday." (`NN` for noun or `VB` for
verb).

In [ ]:
guess("nltk_book_tag", None)   # "NN" or "VB" 

In [ ]:
a = nltk.pos_tag(nltk.word_tokenize("Please book the engineer for Friday."))
b = nltk.pos_tag(nltk.word_tokenize("I read the book on the train."))
print(a)
print(b)
reveal("nltk_book_tag", dict(a)["book"])

`NN`, a noun, in the sentence where it is plainly a verb. NLTK's tagger is an averaged perceptron trained
on newspaper text, where "book" after "Please" is rare and "book" as a noun is common, so the word's own
history outvoted its neighbour. spaCy's tagger, trained on a wider mix of text and using more context, gets
it right, and gives each token a coarse universal tag (`pos_`) beside the fine Penn tag (`tag_`):

In [ ]:
for t in nlp("Please book the engineer for Friday."):
    print(f"{t.text:10} {t.tag_:5} {t.pos_}")

## 3. Chunking: from tags to phrases

A **chunker** groups tagged words into phrases by pattern, without building a full parse; the book calls
it a shallow parser. This grammar says a noun phrase is an optional determiner, any adjectives, then one or
more nouns:

In [ ]:
grammar = "NP: {<DT>?<JJ>*<NN.*>+}"
chunker = nltk.RegexpParser(grammar)
tree = chunker.parse(nltk.pos_tag(nltk.word_tokenize("The replacement Nimbus X2 you sent does not charge.")))
print([" ".join(w for w, _ in st) for st in tree.subtrees() if st.label() == "NP"])

One phrase, "The replacement Nimbus X2": the thing the customer is complaining about, found by a
one-line rule. spaCy offers the same idea ready-made as `doc.noun_chunks`.

## 4. Named entities, and the order of the pipeline

**Named entity recognition** (NER) finds the spans that name something: people, organisations, places,
dates, amounts. Here is the second notice, on the price change:

In [ ]:
price = notices[1]
print(price)

def nltk_entities(tokens):
    tree = nltk.ne_chunk(nltk.pos_tag(tokens))
    return [(" ".join(w for w, _ in st), st.label()) for st in tree if hasattr(st, "label")]

print("NLTK :", nltk_entities(nltk.word_tokenize(price)))
print("spaCy:", [(e.text, e.label_) for e in nlp(price).ents])

NLTK labels the company Kittiwake a PERSON, and the town Port Ember a PERSON too. spaCy gets the person,
the organisation and both towns (`GPE`, a geopolitical entity) right, and also finds the dates and the money,
which NLTK's chunker has no labels for.

The book's Chapter 1 notebook removed stop words and punctuation **before** running NER. Predict what that
does here: will the entities get better, stay the same or get worse?

In [ ]:
guess("ner_after_cleaning", None)   # "better", "same" or "worse" 

In [ ]:
cleaned = [w for w in nltk.word_tokenize(price) if w.lower() not in sw and w.isalnum()]
print(cleaned)
print("NLTK on cleaned :", nltk_entities(cleaned))
print("spaCy on cleaned:", [(e.text, e.label_) for e in nlp(" ".join(cleaned)).ents])
print("NLTK on lowercase:", nltk_entities([w.lower() for w in nltk.word_tokenize(price)]))
reveal("ner_after_cleaning", "worse")

Worse, and in a specific way. With the commas and small words gone, the town Saltmoor and the town Port Ember
run into each other and come out as one PERSON, "Saltmoor Port", and the head of pricing merges with the
company she works for. Lowercase the text and NLTK finds nothing at all, because capital letters were most
of its evidence. That is what happened in the book's notebook: "Miami Dade College", with the words around
it stripped away, was tagged PERSON.

The rule this gives you: **entity recognition runs on the raw text**, before any cleaning. Cleaning is for
the steps that count words; recognition needs every clue the writer left.

## 5. Your turn: every entity in every notice

For each of the five notices, find the entities with spaCy on the **raw** notice, as `[text, label]` pairs,
and save the list of lists.

In [ ]:
all_entities = []
for n in notices:
    ents = []   # YOUR CODE HERE: the [text, label] pairs spaCy finds in n
    all_entities.append(ents)

os.makedirs("out", exist_ok=True)
json.dump(all_entities, open("out/01_03_entities.json", "w"), indent=1)
check_01_03()

In [ ]:
Counter(label for ents in all_entities for _, label in ents).most_common()

Look through what you saved and you will find spaCy's small model making mistakes of its own: "Calls"
labelled a PERSON, "the Aster Fold" labelled a LAW. A small model meeting invented names is a hard test.

## 6. Teaching the pipeline Kittiwake's own names

A statistical model only knows the labels it was trained with, and nobody trained it on Kittiwake's
handsets. Predict: what does spaCy label "Nimbus X2" in this sentence?

In [ ]:
ticket = "The Nimbus X2 I bought on the Unlimited Plus plan won't charge, and my Corvid 5 Pro is fine."
guess("nimbus_label", None)   # a label such as "ORG", or "none" if you think it finds nothing

In [ ]:
ents = [(e.text, e.label_) for e in nlp(ticket).ents]
print(ents)
reveal("nimbus_label", next((l for t, l in ents if "Nimbus" in t), "none"))

It finds nothing for the Nimbus X2 and calls the Corvid 5 Pro a WORK_OF_ART, the label for titles of books and
songs. The fix in production is not a bigger model; it is a few **rules**. spaCy's `EntityRuler` matches
patterns over tokens and, placed before the statistical recogniser, labels what it matches, leaving everything
else to the model. This is the hybrid of rules and learning that Chapter 3 describes, and it is how most real
entity extraction is built.

In [ ]:
shop = spacy.load("en_core_web_sm")
ruler = shop.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([
    # a handset: a family name, then an optional model code such as X2 or 5, then an optional "Pro"
    {"label": "HANDSET", "pattern": [{"LOWER": {"IN": ["nimbus", "corvid", "aster"]}},
                                     {"TEXT": {"REGEX": "^([A-Z]?[0-9]+|Mini|Fold)$"}, "OP": "?"},
                                     {"LOWER": "pro", "OP": "?"}]},
    # a plan: one of the plan families, then an optional word or number
    {"label": "PLAN", "pattern": [{"LOWER": {"IN": ["flex", "unlimited", "family", "prepay"]}},
                                  {"LOWER": {"IN": ["plus", "share", "go"]}, "OP": "?"},
                                  {"LIKE_NUM": True, "OP": "?"}]},
])
print([(e.text, e.label_) for e in shop(ticket).ents])

Both handsets and the plan, labelled with Kittiwake's own labels, in a dozen lines. Each pattern is a list of
conditions, one per token: `LOWER` compares the lowercased text, `REGEX` matches a regular expression, and
`"OP": "?"` makes a token optional. The rules are predictable and easy to audit; the model covers everything
the rules did not think of. You will use the plain pipeline, without these rules, in Part 3.

## 7. N-grams and skip-grams

An **n-gram** is a run of *n* consecutive tokens. Predict how many bigrams (n = 2) there are in
"Practice makes man perfect".

In [ ]:
guess("bigrams", None)

In [ ]:
w = "Practice makes man perfect".split()
print(list(ngrams(w, 2)))
reveal("bigrams", len(list(ngrams(w, 2))))
print("trigrams:", list(ngrams(w, 3)))
print("1-skip bigrams:", list(skipgrams(w, 2, 1)))

Three: *n* tokens give *n - 1* bigrams and *n - 2* trigrams. A **skip-gram** also pairs words that have
other words between them, here up to one; it is how Chapter 5's word2vec learns which words keep company.

## 8. Exit ticket

**x1.** How many bigrams in a four-word sentence? (a) 4, (b) 3, (c) 2

**x3.** When should entity recognition run? (a) after stop words are removed, (b) after lowercasing,
(c) on the raw text, before any cleaning

In [ ]:
ask("x1", "")
ask("x3", "")